# Grok-rl-03-mc-td-qlearning

**Stage 03 — Model-Free Prediction & Control**

## 概念
真实世界常常 **没有 P(s'|s,a)**。只能靠采样轨迹学习。

| 方法 | 更新依据 | 特点 |
|------|----------|------|
| Monte Carlo | 完整回报 G_t | 无偏、高方差、需到 episode 结束 |
| TD(0) | r + γV(s') | 有偏、低方差、可在线 |
| SARSA | on-policy Q | 学“自己会执行”的政策 |
| Q-Learning | off-policy max | 学最优，更敢贴悬崖 |

## 环境
与 Stage02 同构的 Cliff GridWorld，但 **交互采样**，不用模型。


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED = 7
rng = np.random.default_rng(SEED)

gpu_info = {"cuda": False, "device_count": 0, "names": []}
try:
    import torch
    gpu_info = {"cuda": torch.cuda.is_available(), "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
                "names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
except Exception as e:
    gpu_info["error"] = str(e)
print(gpu_info)


In [ ]:

ACTS = {0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
class CliffEnv:
    def __init__(self, slip=0.1, rng=None):
        self.H,self.W=4,12
        self.slip=slip
        self.rng = rng or np.random.default_rng()
        self.start=(3,0); self.goal=(3,11)
        self.cliff={(3,c) for c in range(1,11)}
        self.nS=self.H*self.W; self.nA=4
        self.reset()
    def sid(self,r,c): return r*self.W+c
    def reset(self):
        self.s=self.start; return self.sid(*self.s)
    def step(self,a):
        if self.rng.random()<self.slip:
            a=int(self.rng.integers(0,4))
        r,c=self.s
        dr,dc=ACTS[a]
        nr,nc=r+dr,c+dc
        if not (0<=nr<self.H and 0<=nc<self.W):
            nr,nc=r,c
        if (nr,nc) in self.cliff:
            self.s=self.start
            return self.sid(*self.s), -100.0, True, {}
        if (nr,nc)==self.goal:
            self.s=(nr,nc)
            return self.sid(*self.s), 10.0, True, {}
        self.s=(nr,nc)
        return self.sid(*self.s), -1.0, False, {}

def eps_greedy(Q, s, eps, rng):
    if rng.random()<eps:
        return int(rng.integers(0,Q.shape[1]))
    row=Q[s]; m=row.max(); return int(rng.choice(np.flatnonzero(row==m)))

def run_sarsa(env, episodes=800, alpha=0.5, gamma=0.99, eps=0.1, seed=0):
    rng=np.random.default_rng(seed)
    Q=np.zeros((env.nS, env.nA))
    returns=[]
    for ep in range(episodes):
        s=env.reset(); a=eps_greedy(Q,s,eps,rng)
        G=0.0; done=False; steps=0
        while not done and steps<500:
            ns,r,done,_=env.step(a)
            na = 0 if done else eps_greedy(Q,ns,eps,rng)
            target = r + (0 if done else gamma*Q[ns,na])
            Q[s,a] += alpha*(target - Q[s,a])
            G += r; s,a = ns,na; steps+=1
        returns.append(G)
    return Q, np.array(returns)

def run_qlearning(env, episodes=800, alpha=0.5, gamma=0.99, eps=0.1, seed=0):
    rng=np.random.default_rng(seed)
    Q=np.zeros((env.nS, env.nA))
    returns=[]
    for ep in range(episodes):
        s=env.reset(); G=0.0; done=False; steps=0
        while not done and steps<500:
            a=eps_greedy(Q,s,eps,rng)
            ns,r,done,_=env.step(a)
            target = r + (0 if done else gamma*Q[ns].max())
            Q[s,a] += alpha*(target - Q[s,a])
            G += r; s = ns; steps+=1
        returns.append(G)
    return Q, np.array(returns)

def mc_control_every_visit(env, episodes=800, gamma=0.99, eps=0.2, seed=0):
    rng=np.random.default_rng(seed)
    Q=np.zeros((env.nS, env.nA))
    returns_sum=np.zeros_like(Q); returns_n=np.zeros_like(Q)
    hist=[]
    for ep in range(episodes):
        # generate episode
        s=env.reset(); traj=[]; done=False; steps=0; Gep=0.0
        while not done and steps<500:
            a=eps_greedy(Q,s,eps,rng)
            ns,r,done,_=env.step(a)
            traj.append((s,a,r)); Gep+=r; s=ns; steps+=1
        hist.append(Gep)
        # every-visit MC
        G=0.0
        for s,a,r in reversed(traj):
            G = r + gamma*G
            returns_sum[s,a]+=G; returns_n[s,a]+=1
            Q[s,a]=returns_sum[s,a]/returns_n[s,a]
    return Q, np.array(hist)

def smooth(x, w=20):
    if len(x)<w: return x
    c=np.cumsum(np.insert(x,0,0)); return (c[w:]-c[:-w])/w


In [ ]:

t0=time.time()
env = CliffEnv(slip=0.1, rng=rng)
Qs, ret_s = run_sarsa(env, episodes=1000, seed=1)
Qq, ret_q = run_qlearning(env, episodes=1000, seed=1)
Qm, ret_m = mc_control_every_visit(env, episodes=1000, seed=1)
elapsed=time.time()-t0

def mean_last(x,n=100): return float(np.mean(x[-n:]))
print("SARSA last100", mean_last(ret_s))
print("Qlearn last100", mean_last(ret_q))
print("MC last100", mean_last(ret_m))

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(smooth(ret_s), label="SARSA")
ax.plot(smooth(ret_q), label="Q-Learning")
ax.plot(smooth(ret_m), label="Every-visit MC")
ax.set_xlabel("episode"); ax.set_ylabel("return (smoothed)")
ax.set_title("Model-free control on CliffWorld")
ax.legend(); fig.tight_layout()
fig.savefig(OUT/"stage03_learning_curves.png", dpi=120); plt.close(fig)

def policy_map(Q, env):
    names=["U","R","D","L"]
    rows=[]
    for r in range(env.H):
        row=[]
        for c in range(env.W):
            if (r,c) in env.cliff: row.append("C")
            elif (r,c)==env.goal: row.append("G")
            else:
                s=env.sid(r,c); row.append(names[int(np.argmax(Q[s]))])
        rows.append(" ".join(row))
    return rows

maps = {"SARSA": policy_map(Qs, env), "Q-Learning": policy_map(Qq, env), "MC": policy_map(Qm, env)}
for k,v in maps.items():
    print("==", k); print("\n".join(v))

payload={
  "ok": True,
  "stage":"03-mc-td-qlearning",
  "title":"Grok-rl-03-mc-td-qlearning",
  "metrics":{
    "sarsa_last100": mean_last(ret_s),
    "qlearning_last100": mean_last(ret_q),
    "mc_last100": mean_last(ret_m),
  },
  "policies": maps,
  "gpu": gpu_info,
  "elapsed_sec": elapsed,
  "concept": "learn value/policy from sampled transitions without knowing P",
  "new_capability": "on-policy (SARSA, safer) vs off-policy (Q-learning, optimal) control from interaction",
  "compare_to_previous": "Stage02 needed full model; Stage03 only needs env.step samples",
}
# Q-learning or SARSA should learn something better than -500 average early
assert max(payload["metrics"].values()) > -50
(OUT/"results_stage03.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2)[:1500])
print("STAGE03_OK")
